# 신규 기능이 기존 구매자의 구매 횟수에 미친 영향 분석

## 분석 개요
- **목적**: Daily Tab 기능(10/12 출시)이 기존 구매자의 구매 횟수에 미친 영향 측정
- **방법**: DID(Difference-in-Differences) 분석
- **기간**: 
  - Before: 2025년 8월 1일 ~ 9월 30일
  - After: 2025년 10월 14일 ~ 10월 29일 (프로모션 기간 제외)

## 1. 환경 설정 및 라이브러리 임포트

In [ ]:
# 필요한 라이브러리 임포트
import pandas as pd
import numpy as np
from google.cloud import bigquery
from google.oauth2 import service_account
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# 한글 폰트 설정
plt.rcParams['font.family'] = 'AppleGothic'
plt.rcParams['axes.unicode_minus'] = False

# 스타일 설정
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

In [ ]:
# BigQuery 클라이언트 설정
# credentials = service_account.Credentials.from_service_account_file('path_to_key.json')
# client = bigquery.Client(credentials=credentials, project='hellobot-f445c')

client = bigquery.Client(project='hellobot-f445c')
print("BigQuery 클라이언트 초기화 완료")

## 2. 데이터 추출

### 2.1 기존 구매자 식별 및 그룹 분류

In [ ]:
# 기존 구매자 및 그룹 분류 쿼리
query_user_groups = """
WITH existing_buyers AS (
  SELECT DISTINCT user_id
  FROM `hellobot-f445c.temporary.tony_analysis_daily_fortune_20251029`
  WHERE event_date < '2025-10-12'
    AND event_name LIKE '%pay_for_%'
    AND revenue_krw > 0
),

feature_users AS (
  SELECT DISTINCT user_id
  FROM `hellobot-f445c.temporary.tony_analysis_daily_fortune_20251029`
  WHERE event_date >= '2025-10-12'
    AND viewed_daily_tab = 1
)

SELECT
  eb.user_id,
  CASE
    WHEN fu.user_id IS NOT NULL THEN 'Treatment'
    ELSE 'Control'
  END AS user_group
FROM existing_buyers eb
LEFT JOIN feature_users fu ON eb.user_id = fu.user_id
"""

# 쿼리 실행
df_user_groups = client.query(query_user_groups).to_dataframe()
print(f"총 기존 구매자 수: {len(df_user_groups):,}명")
print(f"\n그룹별 사용자 수:")
print(df_user_groups['user_group'].value_counts())
print(f"\nTreatment 비율: {(df_user_groups['user_group']=='Treatment').mean():.1%}")

### 2.2 사용자별 구매 데이터 추출

In [ ]:
# DID 분석용 사용자별 구매 데이터
query_user_purchases = """
WITH existing_buyers AS (
  SELECT DISTINCT user_id
  FROM `hellobot-f445c.temporary.tony_analysis_daily_fortune_20251029`
  WHERE event_date < '2025-10-12'
    AND event_name LIKE '%pay_for_%'
    AND revenue_krw > 0
),

feature_users AS (
  SELECT DISTINCT user_id
  FROM `hellobot-f445c.temporary.tony_analysis_daily_fortune_20251029`
  WHERE event_date >= '2025-10-12'
    AND viewed_daily_tab = 1
),

user_groups AS (
  SELECT
    eb.user_id,
    CASE
      WHEN fu.user_id IS NOT NULL THEN 'Treatment'
      ELSE 'Control'
    END AS user_group
  FROM existing_buyers eb
  LEFT JOIN feature_users fu ON eb.user_id = fu.user_id
)

SELECT
  ug.user_id,
  ug.user_group,
  
  -- Before Period (8/1 ~ 9/30)
  COUNT(DISTINCT CASE
    WHEN t.event_date BETWEEN '2025-08-01' AND '2025-09-30'
    THEN t.transaction_id
  END) as before_purchases,
  
  COUNT(DISTINCT CASE
    WHEN t.event_date BETWEEN '2025-08-01' AND '2025-09-30'
    THEN DATE(t.event_date)
  END) as before_active_days,
  
  SUM(CASE
    WHEN t.event_date BETWEEN '2025-08-01' AND '2025-09-30'
    THEN t.revenue_krw
    ELSE 0
  END) as before_revenue,
  
  -- After Period (10/14 ~ 10/29)
  COUNT(DISTINCT CASE
    WHEN t.event_date BETWEEN '2025-10-14' AND '2025-10-29'
    THEN t.transaction_id
  END) as after_purchases,
  
  COUNT(DISTINCT CASE
    WHEN t.event_date BETWEEN '2025-10-14' AND '2025-10-29'
    THEN DATE(t.event_date)
  END) as after_active_days,
  
  SUM(CASE
    WHEN t.event_date BETWEEN '2025-10-14' AND '2025-10-29'
    THEN t.revenue_krw
    ELSE 0
  END) as after_revenue

FROM user_groups ug
LEFT JOIN `hellobot-f445c.temporary.tony_analysis_daily_fortune_20251029` t
  ON ug.user_id = t.user_id
  AND t.event_name LIKE '%pay_for_%'
  AND t.revenue_krw > 0
  AND (
    t.event_date BETWEEN '2025-08-01' AND '2025-09-30'
    OR t.event_date BETWEEN '2025-10-14' AND '2025-10-29'
  )
GROUP BY ug.user_id, ug.user_group
HAVING before_purchases > 0  -- Before 기간에 구매 이력이 있는 사용자만
"""

# 쿼리 실행
df_purchases = client.query(query_user_purchases).to_dataframe()
print(f"분석 대상 사용자: {len(df_purchases):,}명")
print(f"\n그룹별 사용자:")
print(df_purchases['user_group'].value_counts())

## 3. 데이터 전처리 및 파생 변수 생성

In [ ]:
# 파생 변수 생성
df_purchases['before_daily_rate'] = df_purchases['before_purchases'] / df_purchases['before_active_days'].replace(0, np.nan)
df_purchases['after_daily_rate'] = df_purchases['after_purchases'] / df_purchases['after_active_days'].replace(0, np.nan)

df_purchases['purchase_change'] = df_purchases['after_purchases'] - df_purchases['before_purchases']
df_purchases['daily_rate_change'] = df_purchases['after_daily_rate'] - df_purchases['before_daily_rate']

df_purchases['purchase_change_rate'] = (
    (df_purchases['after_purchases'] - df_purchases['before_purchases']) / 
    df_purchases['before_purchases'].replace(0, np.nan) * 100
)

df_purchases['revenue_change'] = df_purchases['after_revenue'] - df_purchases['before_revenue']
df_purchases['revenue_change_rate'] = (
    (df_purchases['after_revenue'] - df_purchases['before_revenue']) / 
    df_purchases['before_revenue'].replace(0, np.nan) * 100
)

print("파생 변수 생성 완료")
print(f"\n데이터 shape: {df_purchases.shape}")
print(f"\n컬럼 목록:")
print(df_purchases.columns.tolist())

## 4. 기초 통계 분석

### 4.1 그룹별 Before/After 요약 통계

In [ ]:
# 그룹별 요약 통계
summary_stats = df_purchases.groupby('user_group').agg({
    'before_purchases': ['count', 'mean', 'std', 'median'],
    'after_purchases': ['mean', 'std', 'median'],
    'purchase_change': ['mean', 'std', 'median'],
    'purchase_change_rate': ['mean', 'std', 'median']
}).round(2)

print("=" * 80)
print("그룹별 구매 횟수 통계")
print("=" * 80)
print(summary_stats)

# 더 읽기 쉬운 형태로 정리
for group in ['Treatment', 'Control']:
    group_data = df_purchases[df_purchases['user_group'] == group]
    print(f"\n{group} Group:")
    print(f"  Before 평균 구매: {group_data['before_purchases'].mean():.2f} 회")
    print(f"  After 평균 구매: {group_data['after_purchases'].mean():.2f} 회")
    print(f"  평균 변화량: {group_data['purchase_change'].mean():.2f} 회")
    print(f"  평균 변화율: {group_data['purchase_change_rate'].mean():.1f}%")

### 4.2 DID (Difference-in-Differences) 분석

In [ ]:
# DID 계산
treatment_before = df_purchases[df_purchases['user_group']=='Treatment']['before_purchases'].mean()
treatment_after = df_purchases[df_purchases['user_group']=='Treatment']['after_purchases'].mean()
control_before = df_purchases[df_purchases['user_group']=='Control']['before_purchases'].mean()
control_after = df_purchases[df_purchases['user_group']=='Control']['after_purchases'].mean()

treatment_diff = treatment_after - treatment_before
control_diff = control_after - control_before
did_effect = treatment_diff - control_diff

print("=" * 60)
print("Difference-in-Differences (DID) 분석 결과")
print("=" * 60)
print(f"\nTreatment Group:")
print(f"  Before: {treatment_before:.2f} 회")
print(f"  After:  {treatment_after:.2f} 회")
print(f"  변화량: {treatment_diff:.2f} 회 ({treatment_diff/treatment_before*100:.1f}%)")

print(f"\nControl Group:")
print(f"  Before: {control_before:.2f} 회")
print(f"  After:  {control_after:.2f} 회")
print(f"  변화량: {control_diff:.2f} 회 ({control_diff/control_before*100:.1f}%)")

print(f"\n" + "="*40)
print(f"DID 효과: {did_effect:.2f} 회")
print(f"효과 크기: {did_effect/control_before*100:.1f}% (Control Before 대비)")
print("="*40)

## 5. 통계적 검정

### 5.1 정규성 검정

In [ ]:
# 정규성 검정 (Shapiro-Wilk test)
from scipy.stats import shapiro

print("정규성 검정 (Shapiro-Wilk Test)")
print("="*50)

for group in ['Treatment', 'Control']:
    group_data = df_purchases[df_purchases['user_group'] == group]['purchase_change'].dropna()
    
    # 샘플 크기가 너무 크면 5000개만 샘플링
    if len(group_data) > 5000:
        group_data = group_data.sample(5000, random_state=42)
    
    stat, p_value = shapiro(group_data)
    print(f"{group} Group: p-value = {p_value:.6f}")
    if p_value < 0.05:
        print(f"  → 정규분포를 따르지 않음 (p < 0.05)")
    else:
        print(f"  → 정규분포를 따름 (p >= 0.05)")
    print()

### 5.2 Mann-Whitney U Test (비모수 검정)

In [ ]:
# Mann-Whitney U test
from scipy.stats import mannwhitneyu

treatment_changes = df_purchases[df_purchases['user_group']=='Treatment']['purchase_change'].dropna()
control_changes = df_purchases[df_purchases['user_group']=='Control']['purchase_change'].dropna()

statistic, p_value = mannwhitneyu(treatment_changes, control_changes, alternative='two-sided')

print("Mann-Whitney U Test 결과")
print("="*50)
print(f"통계량: {statistic:,.0f}")
print(f"p-value: {p_value:.6f}")

if p_value < 0.05:
    print("\n결론: 두 그룹 간 구매 횟수 변화에 통계적으로 유의한 차이가 있음")
else:
    print("\n결론: 두 그룹 간 구매 횟수 변화에 통계적으로 유의한 차이가 없음")

# Effect Size (r) 계산
n1, n2 = len(treatment_changes), len(control_changes)
z_score = (statistic - n1*n2/2) / np.sqrt(n1*n2*(n1+n2+1)/12)
effect_size = abs(z_score) / np.sqrt(n1 + n2)
print(f"\nEffect Size (r): {effect_size:.3f}")
if effect_size < 0.1:
    print("  → 매우 작은 효과")
elif effect_size < 0.3:
    print("  → 작은 효과")
elif effect_size < 0.5:
    print("  → 중간 효과")
else:
    print("  → 큰 효과")

## 6. 시각화

### 6.1 Before/After 구매 횟수 분포

In [ ]:
# Before/After 분포 비교
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Treatment - Before
treatment_data = df_purchases[df_purchases['user_group']=='Treatment']
axes[0,0].hist(treatment_data['before_purchases'], bins=30, alpha=0.7, color='blue', edgecolor='black')
axes[0,0].axvline(treatment_data['before_purchases'].mean(), color='red', linestyle='--', label=f'평균: {treatment_data["before_purchases"].mean():.1f}')
axes[0,0].set_title('Treatment Group - Before Period')
axes[0,0].set_xlabel('구매 횟수')
axes[0,0].set_ylabel('사용자 수')
axes[0,0].legend()

# Treatment - After
axes[0,1].hist(treatment_data['after_purchases'], bins=30, alpha=0.7, color='green', edgecolor='black')
axes[0,1].axvline(treatment_data['after_purchases'].mean(), color='red', linestyle='--', label=f'평균: {treatment_data["after_purchases"].mean():.1f}')
axes[0,1].set_title('Treatment Group - After Period')
axes[0,1].set_xlabel('구매 횟수')
axes[0,1].set_ylabel('사용자 수')
axes[0,1].legend()

# Control - Before
control_data = df_purchases[df_purchases['user_group']=='Control']
axes[1,0].hist(control_data['before_purchases'], bins=30, alpha=0.7, color='blue', edgecolor='black')
axes[1,0].axvline(control_data['before_purchases'].mean(), color='red', linestyle='--', label=f'평균: {control_data["before_purchases"].mean():.1f}')
axes[1,0].set_title('Control Group - Before Period')
axes[1,0].set_xlabel('구매 횟수')
axes[1,0].set_ylabel('사용자 수')
axes[1,0].legend()

# Control - After
axes[1,1].hist(control_data['after_purchases'], bins=30, alpha=0.7, color='green', edgecolor='black')
axes[1,1].axvline(control_data['after_purchases'].mean(), color='red', linestyle='--', label=f'평균: {control_data["after_purchases"].mean():.1f}')
axes[1,1].set_title('Control Group - After Period')
axes[1,1].set_xlabel('구매 횟수')
axes[1,1].set_ylabel('사용자 수')
axes[1,1].legend()

plt.tight_layout()
plt.show()

### 6.2 구매 변화량 분포 비교

In [ ]:
# 구매 변화량 박스플롯
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# 박스플롯
bp_data = [treatment_changes, control_changes]
bp = axes[0].boxplot(bp_data, labels=['Treatment', 'Control'], patch_artist=True)
bp['boxes'][0].set_facecolor('lightblue')
bp['boxes'][1].set_facecolor('lightgray')
axes[0].set_title('구매 횟수 변화량 분포')
axes[0].set_ylabel('구매 횟수 변화량')
axes[0].grid(True, alpha=0.3)
axes[0].axhline(y=0, color='red', linestyle='--', alpha=0.5)

# 바이올린 플롯
parts = axes[1].violinplot(bp_data, positions=[0, 1], showmeans=True, showmedians=True)
axes[1].set_xticks([0, 1])
axes[1].set_xticklabels(['Treatment', 'Control'])
axes[1].set_title('구매 횟수 변화량 분포 (Violin Plot)')
axes[1].set_ylabel('구매 횟수 변화량')
axes[1].grid(True, alpha=0.3)
axes[1].axhline(y=0, color='red', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

### 6.3 DID 시각화

In [ ]:
# DID 그래프
fig, ax = plt.subplots(1, 1, figsize=(10, 6))

# 데이터 준비
periods = ['Before\n(8/1~9/30)', 'After\n(10/14~10/29)']
treatment_values = [treatment_before, treatment_after]
control_values = [control_before, control_after]

# 라인 플롯
ax.plot(periods, treatment_values, 'o-', color='blue', linewidth=2, markersize=8, label='Treatment Group')
ax.plot(periods, control_values, 's-', color='red', linewidth=2, markersize=8, label='Control Group')

# 반투명 영역으로 DID 효과 표시
ax.fill_between([0, 1], 
                [treatment_before, treatment_after],
                [control_before, control_after],
                alpha=0.2, color='green', label=f'DID 효과: {did_effect:.2f}')

# 변화량 텍스트 추가
ax.annotate(f'변화: {treatment_diff:.2f}', 
            xy=(1, treatment_after), xytext=(1.1, treatment_after),
            fontsize=10, color='blue')
ax.annotate(f'변화: {control_diff:.2f}', 
            xy=(1, control_after), xytext=(1.1, control_after),
            fontsize=10, color='red')

ax.set_title('Difference-in-Differences (DID) 분석', fontsize=14, fontweight='bold')
ax.set_ylabel('평균 구매 횟수', fontsize=12)
ax.set_xlabel('기간', fontsize=12)
ax.legend(loc='best')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 7. 일별 추이 분석

In [ ]:
# 일별 구매 추이 데이터 조회
query_daily_trend = """
WITH existing_buyers AS (
  SELECT DISTINCT user_id
  FROM `hellobot-f445c.temporary.tony_analysis_daily_fortune_20251029`
  WHERE event_date < '2025-10-12'
    AND event_name LIKE '%pay_for_%'
    AND revenue_krw > 0
),

feature_users AS (
  SELECT DISTINCT user_id
  FROM `hellobot-f445c.temporary.tony_analysis_daily_fortune_20251029`
  WHERE event_date >= '2025-10-12'
    AND viewed_daily_tab = 1
),

user_groups AS (
  SELECT
    eb.user_id,
    CASE
      WHEN fu.user_id IS NOT NULL THEN 'Treatment'
      ELSE 'Control'
    END AS user_group
  FROM existing_buyers eb
  LEFT JOIN feature_users fu ON eb.user_id = fu.user_id
)

SELECT
  DATE(t.event_date) as purchase_date,
  ug.user_group,
  COUNT(DISTINCT t.transaction_id) as daily_purchases,
  COUNT(DISTINCT t.user_id) as daily_purchasers,
  AVG(t.revenue_krw) as avg_order_value
FROM user_groups ug
INNER JOIN `hellobot-f445c.temporary.tony_analysis_daily_fortune_20251029` t
  ON ug.user_id = t.user_id
WHERE t.event_name LIKE '%pay_for_%'
  AND t.revenue_krw > 0
  AND (
    t.event_date BETWEEN '2025-08-01' AND '2025-09-30'
    OR t.event_date BETWEEN '2025-10-01' AND '2025-10-29'
  )
GROUP BY DATE(t.event_date), ug.user_group
ORDER BY purchase_date, user_group
"""

df_daily = client.query(query_daily_trend).to_dataframe()
df_daily['purchase_date'] = pd.to_datetime(df_daily['purchase_date'])
print(f"일별 데이터 로드 완료: {len(df_daily)} 행")

In [ ]:
# 일별 추이 그래프
fig, axes = plt.subplots(2, 1, figsize=(15, 10))

# Treatment vs Control 일별 구매 건수
for group in ['Treatment', 'Control']:
    group_data = df_daily[df_daily['user_group'] == group].sort_values('purchase_date')
    axes[0].plot(group_data['purchase_date'], 
                 group_data['daily_purchases'], 
                 label=group, alpha=0.7, linewidth=1.5)

# 주요 날짜 표시
axes[0].axvline(pd.Timestamp('2025-10-01'), color='orange', linestyle='--', alpha=0.5, label='프로모션 시작')
axes[0].axvline(pd.Timestamp('2025-10-12'), color='red', linestyle='--', alpha=0.5, label='신규 기능 출시')
axes[0].axvline(pd.Timestamp('2025-10-13'), color='orange', linestyle='--', alpha=0.5, label='프로모션 종료')

axes[0].set_title('일별 구매 건수 추이', fontsize=14)
axes[0].set_xlabel('날짜')
axes[0].set_ylabel('구매 건수')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# 일별 평균 구매 금액
for group in ['Treatment', 'Control']:
    group_data = df_daily[df_daily['user_group'] == group].sort_values('purchase_date')
    axes[1].plot(group_data['purchase_date'], 
                 group_data['avg_order_value'], 
                 label=group, alpha=0.7, linewidth=1.5)

axes[1].axvline(pd.Timestamp('2025-10-01'), color='orange', linestyle='--', alpha=0.5, label='프로모션 시작')
axes[1].axvline(pd.Timestamp('2025-10-12'), color='red', linestyle='--', alpha=0.5, label='신규 기능 출시')
axes[1].axvline(pd.Timestamp('2025-10-13'), color='orange', linestyle='--', alpha=0.5, label='프로모션 종료')

axes[1].set_title('일별 평균 구매 금액 추이', fontsize=14)
axes[1].set_xlabel('날짜')
axes[1].set_ylabel('평균 구매 금액 (KRW)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 8. 결과 요약 및 인사이트

In [ ]:
# 최종 결과 요약 테이블
summary_table = pd.DataFrame({
    'Metric': ['사용자 수', 'Before 평균 구매', 'After 평균 구매', '평균 변화량', '평균 변화율(%)', 'DID 효과'],
    'Treatment': [
        len(df_purchases[df_purchases['user_group']=='Treatment']),
        treatment_before,
        treatment_after,
        treatment_diff,
        treatment_diff/treatment_before*100,
        did_effect
    ],
    'Control': [
        len(df_purchases[df_purchases['user_group']=='Control']),
        control_before,
        control_after,
        control_diff,
        control_diff/control_before*100,
        0
    ]
})

summary_table = summary_table.round(2)

print("\n" + "="*60)
print("분석 결과 요약")
print("="*60)
print(summary_table.to_string(index=False))

print("\n" + "="*60)
print("주요 인사이트")
print("="*60)

# 통계적 유의성 판단
if p_value < 0.05:
    significance = "통계적으로 유의함"
else:
    significance = "통계적으로 유의하지 않음"

# 효과 방향 판단
if did_effect > 0:
    effect_direction = "긍정적"
else:
    effect_direction = "부정적"

print(f"1. 신규 기능(Daily Tab)의 효과: {effect_direction} ({did_effect:.2f} 회)")
print(f"2. 통계적 유의성: {significance} (p-value: {p_value:.6f})")
print(f"3. Effect Size: {effect_size:.3f} (효과 크기)")
print(f"4. Treatment 그룹이 Control 그룹 대비 {abs(did_effect/control_before*100):.1f}% {'더 많은' if did_effect > 0 else '더 적은'} 구매 증가")

if did_effect > 0:
    print(f"\n결론: 신규 기능이 기존 구매자의 구매 횟수 증가에 긍정적인 영향을 미침")
else:
    print(f"\n결론: 신규 기능이 기존 구매자의 구매 횟수에 부정적이거나 중립적인 영향을 미침")

## 9. 추가 분석 제언

1. **Propensity Score Matching**: 선택 편향 제거를 위한 PSM 분석
2. **세그먼트별 분석**: RFM, 연령대, 플랫폼별 효과 차이 분석
3. **장기 효과 추적**: 더 긴 기간 데이터로 지속성 평가
4. **프로모션 효과 분리**: 더 정교한 프로모션 효과 보정 모델 적용